# PWV03c : Two-Point Temporal Correlation Function — per filter (pyzdcf)

## Overview

This notebook is a variant of `PWV03_TwoPoint_TemporalCorrelation_separateFilters.ipynb`
that implements the two-point temporal correlation function using
**Sylvie Dagoret-Campagne's own estimator**, originally coded in:

- `OLD_PWV_FrequencyFitGPAndMerra2ComparisonMarch2025.ipynb`
- `OLD_PWV_FrequencyFitSinusAndMerra2ComparisonMarch2025.ipynb`

## Statistical estimator

The discrete correlation function is estimated as:
$$
C(\Delta t) =
  \frac{\bigl\langle \delta\mathrm{PWV}(t_i)\,\delta\mathrm{PWV}(t_j)\bigr\rangle_{|\Delta t_{ij} \in \mathrm{bin}|} }
       {\sigma^2_{\mathrm{PWV}}}
\qquad\text{where}\quad \Delta t_{ij} = t_j - t_i > 0
$$

The implementation (`ComputePWVAndTimeDifference`) loops over all ordered pairs
$(i, j)$ with $t_j \geq t_i$, stores their time lag and PWV product, and then
`ComputeMyDCF_PwvixPwvj` bins the products in a user-defined lag grid and
normalises by the global $\sigma^2$.

## Differences with PWV03 reference notebook

- Sections 1–3 (data loading, quality cuts, night-mean subtraction) are **identical**.
- Section 4 uses Sylvie's explicit pair loop + `ComputeMyDCF_PwvixPwvj` binning.
- The nightly mean subtraction now mirrors the per-night approach of PWV03.
- The same exponential fit and per-filter analysis are kept.

## Key design choices carried over from OLD notebooks

- The **mean PWV** used for the normalisation can be taken either globally
  (as in the OLD notebooks) or per-night (as in PWV03); this notebook provides
  both approaches for comparison, defaulting to the per-night subtraction.
- Pairs are **not restricted to the same night** in `ComputePWVAndTimeDifference`:
  any pair with $\Delta t > 0$ is included.  A same-night variant is also provided.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from astropy.time import Time
from scipy.optimize import curve_fit

%matplotlib inline

# ── publication rc params ─────────────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})


In [ ]:
from pyzdcf import pyzdcf

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()


In [ ]:
pathfigs = 'figs_PWV03dcorr'
prefix   = 'pwv03dcorr_sylvie'
os.makedirs(pathfigs, exist_ok=True)
figtype  = '.pdf'


In [ ]:
FLAG_WITHCOLLIMATOR     = False
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')


In [ ]:
SIGMA_repeat = 0.15

---
## 1. Load data & build Time column


👉 ça simule l’ancien module

⚠️ marche souvent, mais pas garanti si d’autres modules ont bougé

In [ ]:
# hack pour la lecture des pickes
import numpy as np
import types
import sys

sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# Time from DATE-OBS (same as PWV01/02)
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values("Time", ascending=True).reset_index(drop=True)


if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec["Time"] > datetime_WITHCOLLIMATOR].copy()
    

print(f"Shape: {df_spec.shape}")
print(f"Date range: {df_spec['Time'].min().date()}  ->  {df_spec['Time'].max().date()}")

In [ ]:
## Re-compute nightObs from Time (id is buggy in Corentin's files)

In [ ]:
df_spec["nightObs"] = df_spec.apply(lambda x: x['id']//100_000, axis=1)
df_spec["seq_num"]  = df_spec["id"] % 100_000
df_spec["night_from_time_utc"] = (
    df_spec["Time"].dt.strftime("%Y%m%d").astype(int)
)

mismatch = df_spec[
    df_spec["night_from_time_utc"] != df_spec["nightObs"]
]

len(mismatch), len(df_spec)

In [ ]:
FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    print("COMPUTE NIGHTOBS FROM Time") 
    # Convertir au Chili
    df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")

    # Decalage de 12h vers le passe pour rattacher les obs apres minuit a la bonne nuit
    df_spec["nightObs_corr"] = (
        (df_spec["Time_local"] - pd.Timedelta(hours=12))
        .dt.strftime("%Y%m%d")
        .astype(int)
    )
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    span = (
        df_spec.groupby("nightObs_corr")["Time"].max()
        - df_spec.groupby("nightObs_corr")["Time"].min()
    )
    print(span.max())
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    df_spec["nightObs"] = df_spec["nightObs_corr"]

In [ ]:
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Quality cuts


In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"   # reuse cuts from PWV01 notebook
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Tight cuts: {len(df_keep)} / {len(df_spec)} kept")

---

In [ ]:
def ComputeZDCF(filename_in,df_pwv_curve,minpts=0):
    """
    Compute the Discrete Covariance Curve with pyzdcf

    parameters :
    - df_pwv_curve : pandas dataframe with 3 columns : (time, pwv, sigma)
      The time has to be chosen in terms of days/hours,min ..., outside this function
    - filename_in : csv file where are written the tempory DTC curve
    
    """


    # add the error on the point 
    #df_pwvc = df_pwvc.assign(sig_pwv = lambda x: sigma_repeatability)

    full_filename_in = os.path.join(dcf_path_input,filename_in)
    df_pwv_curve.to_csv(full_filename_in, index=False,header=False)

    # parameters for the pyzdcf
    params_dcf = dict(autocf    =  True, # Autocorrelation (T) or cross-correlation (F)
              prefix            = 'acf',  # Output files prefix
              uniform_sampling  =  False, # Uniform sampling?
              omit_zero_lags    =  False,  # Omit zero lag points?
              minpts            =  minpts,     # Min. num. of points per bin (0 is a flag for default value of 11)
              num_MC            =  100,   # Num. of Monte Carlo simulations for error estimation
              lc1_name          =  filename_in,   # Name of the first light curve file
              lc2_name          =  filename_in    # Name of the second light curve file (required only if we do CCF)
             )

    # compute the ZDCF
    
    dcf_df = pyzdcf(input_dir  =  dcf_path_input + "/" , 
                    output_dir = dcf_path_output + "/", 
                    intr       = False, 
                    parameters = params_dcf, 
                    sep        = ',', 
                    sparse     = 'auto', 
                    verbose    = False)
    return dcf_df

In [ ]:
def ComputeCrossZDCF(filename_in1,filename_in2,df_pwv_curve1,df_pwv_curve2,minpts=0):
    """
    Compute the Discrete Covariance Curve with pyzdcf

    parameters :
    - df_pwv_curve : pandas dataframe with 3 columns : (time, pwv, sigma)
      The time has to be chosen in terms of days/hours,min ..., outside this function
    - filename_in : csv file where are written the tempory DTC curve
    
    """

    # add the error on the point 
    #df_pwvc = df_pwvc.assign(sig_pwv = lambda x: sigma_repeatability)

    full_filename_in1 = os.path.join(dcf_path_input,filename_in1)
    df_pwv_curve1.to_csv(full_filename_in1, index=False,header=False)

    full_filename_in2 = os.path.join(dcf_path_input,filename_in2)
    df_pwv_curve2.to_csv(full_filename_in2, index=False,header=False)

    # parameters for the pyzdcf
    params_dcf = dict(autocf    =  False, # Autocorrelation (T) or cross-correlation (F)
              prefix            = 'crosscf',  # Output files prefix
              uniform_sampling  =  False, # Uniform sampling?
              omit_zero_lags    =  False,  # Omit zero lag points?
              minpts            =  minpts,     # Min. num. of points per bin (0 is a flag for default value of 11)
              num_MC            =  100,   # Num. of Monte Carlo simulations for error estimation
              lc1_name          =  filename_in1,   # Name of the first light curve file
              lc2_name          =  filename_in2    # Name of the second light curve file (required only if we do CCF)
             )

    # compute the ZDCF
    
    dcf_df = pyzdcf(input_dir  =  dcf_path_input + "/" , 
                    output_dir = dcf_path_output + "/", 
                    intr       = False, 
                    parameters = params_dcf, 
                    sep        = ',', 
                    sparse     = 'auto', 
                    verbose    = False)
    return dcf_df

In [ ]:
pathdata = "data_pyzdcf"
if not os.path.exists(pathdata):
    os.makedirs(pathdata) 

dcf_path_input = os.path.join(pathdata,"dcf_timecurves") 
dcf_path_output = os.path.join(pathdata,"dcf_results") 
if not os.path.exists(dcf_path_input):
    os.makedirs(dcf_path_input) 
if not os.path.exists(dcf_path_output):
    os.makedirs(dcf_path_output) 

---
## 3. Prepare PWV time series — night-by-night mean subtraction


In [ ]:
SIGMA_REPEATABILITY = 0.15

### a) Search for long timescale correlation in PWV Auxtel Data

#### Discrete corvariance on the PWV values

##### Prepare the data with the three required columns

In [ ]:
df_dcf_in = df_keep[["MJD","PWV [mm]_ram"]]

In [ ]:
tstart = df_dcf_in["MJD"].min()
df_dcf_in["t_day"] = df_dcf_in["MJD"] - tstart

In [ ]:
df_dcf_in = df_dcf_in.assign(sig_pwv = lambda x: SIGMA_REPEATABILITY)

In [ ]:
df_dcf_in = df_dcf_in[["t_day","PWV [mm]_ram","sig_pwv"]]

In [ ]:
df_dcf_in.head() 

In [ ]:
df_dcf_out = ComputeZDCF("dcf_in_pwv_auxtel.csv",df_dcf_in, minpts = 21 )
#df_dcf_out = ComputeZDCF("dcf_in_pwv_auxtel.csv",df_dcf_in, minpts = 101 )

In [ ]:
xerr = df_dcf_out[["-sig(tau)","+sig(tau)"]].values.T	
yerr = df_dcf_out[["-err(dcf)","+err(dcf)"]].values.T	
x = df_dcf_out["tau"].values
y = df_dcf_out["dcf"].values

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(12,5),layout="constrained")
ax.errorbar(x,y,xerr=xerr,yerr=yerr,marker='o', mfc='red',linewidth=0.5,
         mec='red', ms=2, mew=2,ecolor="k",elinewidth=2,capsize=2,uplims=True, lolims=True)
ax.grid()
ax.set_ylim(-1.2,1.2)
ax.set_title(f"Discrete covariance function on PWV measurements in Auxtel (holo)")
ax.set_xlabel("Time (days)")
ax.set_ylabel("DCF (no units)")
ax.grid()
plt.show()